In [ ]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import netket as nk
import netket.experimental as nkx
import sys
sys.path.append('..')
from NES_VMC import NESTotalAnsatz, create_machine,\
    SingleStateAnsatz,create_single_machine,\
        create_machine_matrix,Ham_psi,Ham_Psi,NES_loss_energy,nes_vmc_gradient,\
        NESFermionHopRule,compute_qgt,sampler_info
import optax
from typing import Callable
from functools import partial
from jax.flatten_util import ravel_pytree
import time
import itertools
from pyscf import gto, scf, fci
import numpy as np
import logging


jnp.set_printoptions(
    linewidth=9999,
    threshold=jnp.inf,
    precision=8,
    suppress=False,
)

# 开启 x64 提高精度
jax.config.update("jax_enable_x64", True)


bond_length = 1.8
geometry = [
    ("H", (0.0, 0.0, 0.0)),
    ("H", (bond_length, 0.0, 0.0)),
]

mol = gto.M(atom=geometry, basis="6-31G", verbose=0)
mf = scf.RHF(mol).run(verbose=0)
hf_ground_energy = mf.e_tot

cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()

print("=" * 60)
print("H2 / 6-31G 基准")
print("=" * 60)
print(f"HF energy = {hf_ground_energy:.8f} Ha")
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha | excitation = {exc:.4f} eV")

hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=4,
    s=1/2,
    n_fermions_per_spin=(1, 1),
)

K = 2
hi_ext = hi ** K
ha = nkx.operator.from_pyscf_molecule(mol)

SINGLE_SIZE = hi.size

print("=" * 60)
print("Hilbert 信息")
print("=" * 60)
print(f"K = {K}")
print(f"hi.size = {hi.size}")
print(f"hi_ext.size = {hi_ext.size}")
print(f"SINGLE_SIZE = {SINGLE_SIZE}")

target_loss = float(np.sum(E_fcis[:K]))
print(f"target_loss = sum(E_fcis[:K]) = {target_loss:.8f}")


Hatree_Fock = hi.all_states()[0]
alpha_orbs = [0, 1, 2, 3]
beta_orbs  = [4, 5, 6, 7]

single_edges_full = (
    list(itertools.combinations(alpha_orbs, 2))
    + list(itertools.combinations(beta_orbs, 2))
)

ext_edges = []
for k in range(K):
    offset = k * SINGLE_SIZE
    for i, j in single_edges_full:
        ext_edges.append((i + offset, j + offset))

ext_edges = jnp.asarray(ext_edges)

nes_rule = NESFermionHopRule(
    edges=ext_edges,
    K=K,
    single_size=SINGLE_SIZE,
)

print(single_edges_full)

In [ ]:
# ============================================================
# 工具函数（与诊断版相同）
# ============================================================

def tree_l2_norm(tree):
    leaves = jax.tree_util.tree_leaves(tree)
    if len(leaves) == 0:
        return jnp.array(0.0)
    return jnp.sqrt(
        sum([jnp.sum(jnp.abs(x) ** 2) for x in leaves])
    )

def tree_all_finite(tree):
    leaves = jax.tree_util.tree_leaves(tree)
    if len(leaves) == 0:
        return True
    flags = [jnp.all(jnp.isfinite(x)) for x in leaves]
    return bool(jnp.all(jnp.asarray(flags)))

def unique_ratio_from_samples(samples):
    """
    samples: shape (n_samples, hi_ext.size)
    """
    arr = np.asarray(samples)
    arr = arr.reshape(arr.shape[0], -1)
    unique = len({tuple(row.tolist()) for row in arr})
    return unique / max(arr.shape[0], 1)

def tree_batch_std_norm(tree, batch_size):
    """
    计算 dlogΨ(params, x) 在 batch 维度上的变化强度。
    如果这个接近 0，说明不同样本上的参数响应几乎一样，
    covariance 梯度会自然消失。
    """
    leaves = jax.tree_util.tree_leaves(tree)
    total = 0.0

    for leaf in leaves:
        if leaf.ndim >= 1 and leaf.shape[0] == batch_size:
            centered = leaf - jnp.mean(leaf, axis=0, keepdims=True)
            total = total + jnp.sum(jnp.abs(centered) ** 2)

    return jnp.sqrt(total)

def tree_batch_mean_norm(tree, batch_size):
    leaves = jax.tree_util.tree_leaves(tree)
    total = 0.0

    for leaf in leaves:
        if leaf.ndim >= 1 and leaf.shape[0] == batch_size:
            mean_leaf = jnp.mean(leaf, axis=0)
            total = total + jnp.sum(jnp.abs(mean_leaf) ** 2)

    return jnp.sqrt(total)

def safe_real(x):
    return float(jnp.real(x))

def safe_float(x):
    return float(jnp.asarray(x))

In [ ]:
# ============================================================
# 诊断函数
# ============================================================

def compute_diagnostics(
    total_params,
    x_batch,
    samples,
    E_L_mean,
):
    """
    返回当前参数下的完整诊断量。
    """

    n_total = x_batch.shape[0]
    n_diag = min(DIAG_BATCH_SIZE, n_total)

    x_diag = x_batch[:n_diag]
    samples_diag = samples[:n_diag]

    # ---------- logΨ 统计 ----------
    log_Psi_batch = jax.vmap(
        lambda xx: total_machine(total_params, xx)
    )(x_batch)

    log_real = jnp.real(log_Psi_batch)
    log_imag = jnp.imag(log_Psi_batch)

    log_real_mean = jnp.mean(log_real)
    log_real_std = jnp.std(log_real)
    log_real_span = jnp.max(log_real) - jnp.min(log_real)

    log_imag_mean = jnp.mean(log_imag)
    log_imag_std = jnp.std(log_imag)
    log_imag_span = jnp.max(log_imag) - jnp.min(log_imag)

    # ---------- E_L_batch / trace 统计 ----------
    loss_batch, E_L_batch = NES_loss_energy(
        ha=ha,
        total_matrix_machine=total_matrix_machine,
        single_machine_list=single_machine_list,
        total_params=total_params,
        x=x_diag,
    )

    trace_batch = jnp.trace(E_L_batch, axis1=-2, axis2=-1)
    trace_real = jnp.real(trace_batch)
    trace_imag = jnp.imag(trace_batch)

    trace_real_mean = jnp.mean(trace_real)
    trace_real_std = jnp.std(trace_real)
    trace_real_span = jnp.max(trace_real) - jnp.min(trace_real)

    trace_imag_mean = jnp.mean(trace_imag)
    trace_imag_std = jnp.std(trace_imag)
    trace_imag_span = jnp.max(trace_imag) - jnp.min(trace_imag)

    # ---------- E_L_mean Hermiticity ----------
    herm_error = (
        jnp.linalg.norm(E_L_mean - E_L_mean.conj().T)
        / (jnp.linalg.norm(E_L_mean) + 1e-12)
    )

    imag_norm = jnp.linalg.norm(jnp.imag(E_L_mean))

    E_L_herm = 0.5 * (E_L_mean + E_L_mean.conj().T)
    eig_vals_herm = jnp.linalg.eigvalsh(E_L_herm)

    eig_vals_raw = jnp.linalg.eigvals(E_L_mean)
    eig_vals_raw = eig_vals_raw[jnp.argsort(jnp.real(eig_vals_raw))]

    # ---------- sampler unique ratio ----------
    unique_ratio = unique_ratio_from_samples(samples_diag)

    # ---------- dlogΨ batch 方差 ----------
    grad_logPsi = jax.grad(total_machine, argnums=0, holomorphic=True)
    dlogPsi_batch = jax.vmap(
        grad_logPsi,
        in_axes=(None, 0),
    )(total_params, x_diag)

    dlog_std_norm = tree_batch_std_norm(dlogPsi_batch, n_diag)
    dlog_mean_norm = tree_batch_mean_norm(dlogPsi_batch, n_diag)
    dlog_std_ratio = dlog_std_norm / (dlog_mean_norm + 1e-12)

    return {
        "log_Psi_batch": log_Psi_batch,

        "log_real_mean": log_real_mean,
        "log_real_std": log_real_std,
        "log_real_span": log_real_span,

        "log_imag_mean": log_imag_mean,
        "log_imag_std": log_imag_std,
        "log_imag_span": log_imag_span,

        "trace_real_mean": trace_real_mean,
        "trace_real_std": trace_real_std,
        "trace_real_span": trace_real_span,

        "trace_imag_mean": trace_imag_mean,
        "trace_imag_std": trace_imag_std,
        "trace_imag_span": trace_imag_span,

        "herm_error": herm_error,
        "imag_norm": imag_norm,

        "eig_vals_herm": eig_vals_herm,
        "eig_vals_raw": eig_vals_raw,

        "unique_ratio": unique_ratio,

        "dlog_std_norm": dlog_std_norm,
        "dlog_mean_norm": dlog_mean_norm,
        "dlog_std_ratio": dlog_std_ratio,
    }


def log_diagnostics(step, loss_mean, grad_norm_raw, grad_norm_update, grad_norm_clipped, diag):
    eig_vals_herm = diag["eig_vals_herm"]
    eig_vals_raw = diag["eig_vals_raw"]

    energy_herm_str = " | ".join(
        [f"E{i}_herm={eig_vals_herm[i]:.8f}" for i in range(K)]
    )

    energy_raw_str = " | ".join(
        [f"E{i}_raw={eig_vals_raw[i]:.8f}" for i in range(K)]
    )

    logger.info(f"[Step {step:4d}]")
    logger.info(
        f"Loss={loss_mean:.8f} | "
        f"target={target_loss:.8f} | "
        f"gap={float(loss_mean - target_loss):+.8f}"
    )

    logger.info(
        f"Grad | raw={grad_norm_raw:.4e} | "
        f"update={grad_norm_update:.4e} | "
        f"clipped={grad_norm_clipped:.4e} | "
        f"clip_norm={clip_norm:.2e}"
    )

    logger.info(
        "logΨ.real | "
        f"mean={diag['log_real_mean']:.6f} | "
        f"std={diag['log_real_std']:.4e} | "
        f"span={diag['log_real_span']:.4e}"
    )

    logger.info(
        "logΨ.imag | "
        f"mean={diag['log_imag_mean']:.6f} | "
        f"std={diag['log_imag_std']:.4e} | "
        f"span={diag['log_imag_span']:.4e}"
    )

    logger.info(
        "trace(E_L).real | "
        f"mean={diag['trace_real_mean']:.8f} | "
        f"std={diag['trace_real_std']:.4e} | "
        f"span={diag['trace_real_span']:.4e}"
    )

    logger.info(
        "trace(E_L).imag | "
        f"mean={diag['trace_imag_mean']:.8f} | "
        f"std={diag['trace_imag_std']:.4e} | "
        f"span={diag['trace_imag_span']:.4e}"
    )

    logger.info(
        "E_L_mean diagnostics | "
        f"herm_error={diag['herm_error']:.4e} | "
        f"imag_norm={diag['imag_norm']:.4e}"
    )

    logger.info(
        "sampler / dlogΨ | "
        f"unique_ratio={diag['unique_ratio']:.4f} | "
        f"dlog_std_norm={diag['dlog_std_norm']:.4e} | "
        f"dlog_mean_norm={diag['dlog_mean_norm']:.4e} | "
        f"dlog_std_ratio={diag['dlog_std_ratio']:.4e}"
    )

    logger.info(energy_herm_str)
    logger.info(energy_raw_str)

    # 明确报警
    if float(diag["log_real_span"]) < 1e-6:
        logger.warning(">>> WARNING: logΨ.real span ≈ 0，疑似 amplitude collapse / gauge plateau")

    if float(diag["trace_real_std"]) < 1e-8:
        logger.warning(">>> WARNING: trace(E_L).real std 很小，covariance 梯度能量项可能无信号")

    if float(diag["dlog_std_ratio"]) < 1e-6:
        logger.warning(">>> WARNING: dlogΨ batch variation 很小，参数响应接近常数方向")

    if float(diag["herm_error"]) > 1.0:
        logger.warning(">>> WARNING: E_L_mean 非 Hermitian 程度很大，能量诊断不可信")

    logger.info("#" + "-" * 79)

In [ ]:
# ============================================================
# 日志配置
# ============================================================

logger = logging.getLogger(f"NES_VMC_K{K}_diagnostic_original")
logger.setLevel(logging.INFO)
logger.propagate = False
logger.handlers.clear()

simple_formatter = logging.Formatter("%(message)s")

log_filename = f"nes_vmc_H2_631G_K{K}_original_diagnostic.log"

file_handler = logging.FileHandler(log_filename, mode="w", encoding="utf-8")
file_handler.setFormatter(simple_formatter)
file_handler.setLevel(logging.INFO)
logger.addHandler(file_handler)

console_handler = logging.StreamHandler()
console_handler.setFormatter(simple_formatter)
console_handler.setLevel(logging.INFO)
logger.addHandler(console_handler)

logger.info("\n" + "=" * 80)
logger.info("NES-VMC 原版 + Full Edges | Diagnostic Version")
logger.info("=" * 80)
logger.info(f"K = {K}")
logger.info(f"SINGLE_SIZE = {SINGLE_SIZE}")
logger.info(f"target_loss = {target_loss:.8f}")
logger.info(f"FCI E0 = {E_fcis[0]:.8f} | E1 = {E_fcis[1]:.8f}")

In [ ]:
# ============================================================
# 超参数
# ============================================================
N_CHAINS = 16
N_WARMUP = 100
N_SAMPLES_PER_CHAIN = 200
SWEEP_SIZE = 30
N_ITER = 100

Natural_Grad = True

lr = 0.01
clip_norm = 1.0

# 原始梯度超过这个阈值，直接 skip
grad_skip_threshold = 10.0

qgt_diag_shift = 0.1

# 每隔多少步完整诊断一次
PRINT_EVERY = 10

# dlogΨ 诊断比较贵，只取一小部分样本
DIAG_BATCH_SIZE = 128

logger.info("=" * 80)
logger.info("Hyperparameters")
logger.info("=" * 80)
logger.info(f"N_CHAINS = {N_CHAINS}")
logger.info(f"N_WARMUP = {N_WARMUP}")
logger.info(f"N_SAMPLES_PER_CHAIN = {N_SAMPLES_PER_CHAIN}")
logger.info(f"SWEEP_SIZE = {SWEEP_SIZE}")
logger.info(f"N_ITER = {N_ITER}")
logger.info(f"Natural_Grad = {Natural_Grad}")
logger.info(f"lr = {lr}")
logger.info(f"clip_norm = {clip_norm}")
logger.info(f"grad_skip_threshold = {grad_skip_threshold}")
logger.info(f"qgt_diag_shift = {qgt_diag_shift}")
logger.info(f"DIAG_BATCH_SIZE = {DIAG_BATCH_SIZE}")


# ============================================================
# 模型初始化
# ============================================================

total_ansatz = NESTotalAnsatz(SINGLE_SIZE, K, 12, rngs=nnx.Rngs(11))
total_machine, total_graphdef, total_params = create_machine(total_ansatz)
total_matrix_machine, total_graphdef, total_params = create_machine_matrix(total_ansatz)

single_machine_list = []
for ansatz in total_ansatz.single_ansatz_list:
    m, g, p = create_single_machine(ansatz)
    single_machine_list.append(m)


# ============================================================
# Optimizer
# ============================================================

optimizer = optax.chain(
    optax.clip_by_global_norm(clip_norm),
    optax.sgd(learning_rate=lr),
)

opt_state = optimizer.init(total_params)


# ============================================================
# Sampler
# ============================================================

nes_sampler = nk.sampler.MetropolisSampler(
    hilbert=hi_ext,
    rule=nes_rule,
    n_chains=N_CHAINS,
    sweep_size=SWEEP_SIZE,
)

sampler_rng = jax.random.PRNGKey(21)
sampler_state = nes_sampler.init_state(
    total_machine,
    total_params,
    sampler_rng,
)

# warmup
logger.info("=" * 80)
logger.info("Warmup")
logger.info("=" * 80)

for _ in range(N_WARMUP):
    _, sampler_state = nes_sampler.sample(
        machine=total_machine,
        parameters=total_params,
        state=sampler_state,
        chain_length=1,
    )

logger.info("Warmup done.")


# ============================================================
# History
# ============================================================

history = {
    "step": [],
    "loss": [],
    "energies_herm": [],
    "energies_raw": [],
    "E_Lmatrix": [],
    "samples": [],

    "grad_norm_raw": [],
    "grad_norm_update": [],
    "grad_norm_clipped": [],

    "log_real_std": [],
    "log_real_span": [],
    "trace_real_std": [],
    "trace_real_span": [],

    "dlog_std_norm": [],
    "dlog_mean_norm": [],
    "dlog_std_ratio": [],

    "herm_error": [],
    "imag_norm": [],

    "unique_ratio": [],
}

In [ ]:
# ============================================================
# 训练循环（带完整诊断）
# ============================================================

logger.info("\n" + "=" * 80)
logger.info("Start Training")
logger.info("=" * 80)

start_time = time.time()
n_skipped = 0

for step in range(N_ITER):

    # --------------------------------------------------------
    # Sampling
    # --------------------------------------------------------
    samples_raw, sampler_state = nes_sampler.sample(
        machine=total_machine,
        parameters=total_params,
        state=sampler_state,
        chain_length=N_SAMPLES_PER_CHAIN,
    )

    samples = samples_raw.reshape(-1, hi_ext.size)
    x_batch = samples.reshape(-1, K, SINGLE_SIZE)

    # --------------------------------------------------------
    # Gradient
    # --------------------------------------------------------
    grad_raw, loss_mean, E_L_mean = nes_vmc_gradient(
        ha=ha,
        total_matrix_machine=total_matrix_machine,
        total_machine=total_machine,
        single_machine_list=single_machine_list,
        total_params=total_params,
        x_batch=x_batch,
    )

    grad_raw_flat, unravel_fn = ravel_pytree(grad_raw)
    grad_norm_raw = jnp.linalg.norm(grad_raw_flat)

    grad_finite = tree_all_finite(grad_raw)
    loss_finite = bool(jnp.isfinite(loss_mean))
    grad_explode = bool(grad_norm_raw > grad_skip_threshold)

    # --------------------------------------------------------
    # Natural gradient, optional
    # --------------------------------------------------------
    if Natural_Grad:
        qgt_reg_mat, _ = compute_qgt(
            total_machine,
            total_params,
            x_batch,
            diag_shift=qgt_diag_shift,
        )
        ng_flat = jnp.linalg.solve(qgt_reg_mat, grad_raw_flat)
        grad_update = unravel_fn(ng_flat)
        grad_norm_update = jnp.linalg.norm(ng_flat)
    else:
        grad_update = grad_raw
        grad_norm_update = grad_norm_raw

    # clip 后理论范数
    grad_norm_clipped = jnp.minimum(grad_norm_update, clip_norm)

    # --------------------------------------------------------
    # Diagnostics before possible update
    # --------------------------------------------------------
    need_print = (
        step % PRINT_EVERY == 0
        or step == N_ITER - 1
        or grad_explode
        or (not grad_finite)
        or (not loss_finite)
    )

    if need_print:
        diag = compute_diagnostics(
            total_params=total_params,
            x_batch=x_batch,
            samples=samples,
            E_L_mean=E_L_mean,
        )

        log_diagnostics(
            step=step,
            loss_mean=loss_mean,
            grad_norm_raw=grad_norm_raw,
            grad_norm_update=grad_norm_update,
            grad_norm_clipped=grad_norm_clipped,
            diag=diag,
        )

    # --------------------------------------------------------
    # Skip bad update
    # --------------------------------------------------------
    if (not grad_finite) or (not loss_finite) or grad_explode:
        n_skipped += 1
        logger.warning(
            f"【Step {step} Skip】bad update skipped | "
            f"grad_finite={grad_finite} | "
            f"loss_finite={loss_finite} | "
            f"grad_norm_raw={float(grad_norm_raw):.6e} | "
            f"skip_count={n_skipped}"
        )
        logger.info("#" + "=" * 79)
        continue

    # --------------------------------------------------------
    # Optimizer update
    # --------------------------------------------------------
    updates, opt_state = optimizer.update(
        grad_update,
        opt_state,
        total_params,
    )

    total_params = optax.apply_updates(total_params, updates)

    # --------------------------------------------------------
    # Save history
    # --------------------------------------------------------
    if need_print:
        history["step"].append(step)
        history["loss"].append(loss_mean)
        history["energies_herm"].append(diag["eig_vals_herm"])
        history["energies_raw"].append(diag["eig_vals_raw"])
        history["E_Lmatrix"].append(E_L_mean)
        history["samples"].append(samples)

        history["grad_norm_raw"].append(grad_norm_raw)
        history["grad_norm_update"].append(grad_norm_update)
        history["grad_norm_clipped"].append(grad_norm_clipped)

        history["log_real_std"].append(diag["log_real_std"])
        history["log_real_span"].append(diag["log_real_span"])

        history["trace_real_std"].append(diag["trace_real_std"])
        history["trace_real_span"].append(diag["trace_real_span"])

        history["dlog_std_norm"].append(diag["dlog_std_norm"])
        history["dlog_mean_norm"].append(diag["dlog_mean_norm"])
        history["dlog_std_ratio"].append(diag["dlog_std_ratio"])

        history["herm_error"].append(diag["herm_error"])
        history["imag_norm"].append(diag["imag_norm"])

        history["unique_ratio"].append(diag["unique_ratio"])


end_time = time.time()

logger.info("\n" + "=" * 80)
logger.info("Training finished")
logger.info("=" * 80)
logger.info(f"Elapsed time = {end_time - start_time:.2f} sec")
logger.info(f"Skipped updates = {n_skipped}")
logger.info(f"Log file = {log_filename}")

print("\n" + "=" * 80)
print("训练完成")
print("=" * 80)
print(f"训练耗时：{end_time - start_time:.2f} 秒")
print(f"跳过更新次数：{n_skipped}")
print(f"日志文件：{log_filename}")

In [ ]:
# ============================================================
# 可视化训练结果
# ============================================================

import matplotlib.pyplot as plt

fig, axs = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('NES-VMC 原版 + Full Edges | Diagnostic Version | K=2')

# 能量
axs[0, 0].plot(history['energies_raw'])
axs[0, 0].hlines(E_fcis[0], 0, len(history['energies_raw']), linestyle='--', color='red', label='FCI E0')
if K > 1:
    axs[0, 0].hlines(E_fcis[1], 0, len(history['energies_raw']), linestyle='--', color='orange', label='FCI E1')
axs[0, 0].set_title('Eigenvalues')
axs[0, 0].set_xlabel('step')
axs[0, 0].set_ylabel('energy')
axs[0, 0].legend()

# Loss
axs[0, 1].plot(history['loss'])
axs[0, 1].hlines(target_loss, 0, len(history['loss']), linestyle='--', color='red', label='target')
axs[0, 1].set_title('Loss')
axs[0, 1].set_xlabel('step')
axs[0, 1].set_ylabel('loss')
axs[0, 1].legend()

# Grad norm
axs[0, 2].plot(history['grad_norm_raw'], label='raw')
axs[0, 2].plot(history['grad_norm_update'], label='update')
axs[0, 2].plot(history['grad_norm_clipped'], label='clipped')
axs[0, 2].set_title('Grad Norm')
axs[0, 2].set_xlabel('step')
axs[0, 2].set_ylabel('norm')
axs[0, 2].legend()
axs[0, 2].set_yscale('log')

# logΨ span
axs[1, 0].plot(history['log_real_span'])
axs[1, 0].set_title('logΨ.real span')
axs[1, 0].set_xlabel('step')
axs[1, 0].set_ylabel('span')

# trace std
axs[1, 1].plot(history['trace_real_std'])
axs[1, 1].set_title('trace(E_L).real std')
axs[1, 1].set_xlabel('step')
axs[1, 1].set_ylabel('std')
axs[1, 1].set_yscale('log')

# dlog ratio
axs[1, 2].plot(history['dlog_std_ratio'])
axs[1, 2].set_title('dlog std ratio')
axs[1, 2].set_xlabel('step')
axs[1, 2].set_ylabel('ratio')
axs[1, 2].set_yscale('log')

plt.tight_layout()
plt.show()